<p>
<img src="../imgs/EII-ULPGC-logo.jpeg" width="430px" align="right">

# **NOTEBOOK 8.1**
---

# **Function Calling**

Las function calling son una capacidad de los modelos de lenguaje que les permite identificar cuándo una consulta del usuario requiere ejecutar una función externa, generar automáticamente los argumentos estructurados para esa función y devolverlos en un formato que el modelo los pueda interpretar. De este modo, el modelo no solo produce texto, sino que actúa como un orquestador: decide si usar una herramienta, construye los parámetros adecuados y permite que el sistema ejecute la función real (por ejemplo, consultar una API, acceder a una base de datos o realizar un cálculo), integrando después el resultado en la conversación.

https://platform.openai.com/docs/guides/function-calling

<div align="center">
    <img src="imgs/function-calling-diagram-steps.png" width="500px">
</div>
  
### **Ejemplo básico de Function Calling con la API de OpenAI**

In [ ]:

from openai import OpenAI
import json
import os

api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

# ============================================================
# 1. DEFINICIÓN DE LAS HERRAMIENTAS DISPONIBLES
# ============================================================

# Definimos las funciones que el modelo puede solicitar ejecutar.
# IMPORTANTE: aquí solo describimos las herramientas.
# No estamos ejecutando ninguna función todavía.

tools = [
    {
        # Indicamos que la herramienta es una función.
        "type": "function",

        # Nombre de la función que el modelo podrá solicitar.
        "name": "get_horoscope",

        # Descripción de lo que hace la función.
        # El modelo utiliza esta información para decidir cuándo llamarla.
        "description": "Get today's horoscope for an astrological sign.",

        # Definimos los parámetros que acepta la función.
        # Utilizamos JSON Schema para describir su estructura.
        "parameters": {
            "type": "object",

            # Propiedades que puede recibir la función.
            "properties": {
                "sign": {
                    "type": "string",
                    "description": "An astrological sign like Taurus or Aquarius",
                },
            },

            # Indicamos que el parámetro 'sign' es obligatorio.
            "required": ["sign"],
        },
    },
]


# ============================================================
# 2. IMPLEMENTACIÓN REAL DE LA FUNCIÓN
# ============================================================

# Esta es la función Python que ejecutará nuestra aplicación
# cuando el modelo solicite utilizar la herramienta get_horoscope.
#
# El modelo NO ejecuta directamente esta función.
# Nuestra aplicación es responsable de ejecutarla.

def get_horoscope(sign):

    # Devolvemos un horóscopo ficticio para el signo recibido.
    return f"{sign}: Next Tuesday you will befriend a baby otter."


# ============================================================
# 3. PREPARACIÓN DE LA CONVERSACIÓN
# ============================================================

# Creamos una lista que almacenará el historial de la conversación.
#
# Esta lista irá creciendo con:
# - Los mensajes del usuario.
# - Las solicitudes de herramientas realizadas por el modelo.
# - Los resultados obtenidos al ejecutar esas herramientas.

input_list = [
    {
        "role": "user",
        "content": "What is my horoscope? I am an Aquarius."
    }
]


# ============================================================
# 4. PRIMERA LLAMADA AL MODELO
# ============================================================

# Enviamos al modelo:
# - El mensaje del usuario.
# - La descripción de las herramientas disponibles.
#
# Como el usuario pregunta por su horóscopo, el modelo puede
# decidir solicitar una llamada a get_horoscope.

response = client.responses.create(
    model="gpt-5",
    tools=tools,
    input=input_list,
)


# ============================================================
# 5. GUARDAR LA RESPUESTA DEL MODELO
# ============================================================

# Añadimos al historial todos los elementos devueltos
# por el modelo, incluidas las posibles llamadas a funciones.
#
# Esto es importante para mantener el contexto cuando
# volvamos a enviar información al modelo.

input_list += response.output


# ============================================================
# 6. DETECTAR Y EJECUTAR LAS LLAMADAS A FUNCIONES
# ============================================================

# Recorremos los elementos de la respuesta del modelo.
for item in response.output:

    # Comprobamos si el elemento es una solicitud
    # de ejecución de una función.
    if item.type == "function_call":

        # Comprobamos qué función ha solicitado el modelo.
        if item.name == "get_horoscope":

            # Los argumentos llegan como una cadena JSON.
            # Por ejemplo: '{"sign": "Aquarius"}'
            #
            # json.loads() convierte esa cadena en un
            # diccionario de Python.
            arguments = json.loads(item.arguments)

            # Extraemos el signo zodiacal del diccionario.
            sign = arguments["sign"]

            # Ejecutamos la función REAL en nuestro programa.
            # El modelo no ejecuta esta instrucción.
            horoscope = get_horoscope(sign)

            # Preparamos el resultado para enviarlo al modelo.
            #
            # 'function_call_output' indica que este elemento
            # contiene el resultado de una herramienta.
            #
            # 'call_id' permite relacionar este resultado
            # con la solicitud de función correspondiente.
            #
            # json.dumps() convierte un diccionario Python
            # en una cadena JSON.

            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                    "horoscope": horoscope
                })
            })


# ============================================================
# 7. MOSTRAR EL HISTORIAL COMPLETO
# ============================================================

# Mostramos todos los elementos que vamos a enviar
# en la siguiente llamada al modelo.
#
# Ahora la lista contiene:
# - La pregunta original del usuario.
# - La solicitud de ejecución de get_horoscope.
# - El resultado obtenido al ejecutar la función.

print("Final input:")
print(input_list)


# ============================================================
# 8. SEGUNDA LLAMADA AL MODELO
# ============================================================

# Volvemos a consultar al modelo, pero ahora dispone
# del resultado de la función que hemos ejecutado.
#
# La instrucción adicional le indica que debe responder
# únicamente con un horóscopo obtenido mediante una herramienta.

response = client.responses.create(
    model="gpt-5",
    instructions="Respond only with a horoscope generated by a tool.",
    tools=tools,
    input=input_list,
)


# ============================================================
# 9. MOSTRAR LA RESPUESTA FINAL
# ============================================================

# Mostramos la respuesta completa en formato JSON.
# model_dump_json() permite inspeccionar todos los campos.
# indent=2 mejora la legibilidad del JSON.

print("Final output:")
print(response.model_dump_json(indent=2))

# Mostramos únicamente el texto generado por el modelo,
# sin los metadatos ni la estructura JSON de la respuesta.

print("\n" + response.output_text)


Final input:
[{'role': 'user', 'content': 'What is my horoscope? I am an Aquarius.'}, ResponseReasoningItem(id='rs_05de88ec025a636800691f8fdf650881a08d4d0b8860b9d6b0', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseFunctionToolCall(arguments='{"sign":"Aquarius"}', call_id='call_TeW9yCYRkLnuU47ATHljEum9', name='get_horoscope', type='function_call', id='fc_05de88ec025a636800691f8fe39a8c81a081a526c55d7d4090', status='completed'), {'type': 'function_call_output', 'call_id': 'call_TeW9yCYRkLnuU47ATHljEum9', 'output': '{"horoscope": "{\'sign\': \'Aquarius\'}: Next Tuesday you will befriend a baby otter."}'}]
Final output:
{
  "id": "resp_05de88ec025a636800691f8fe474f481a0bcaf7bce6da9d25e",
  "created_at": 1763676132.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with a horoscope generated by a tool.",
  "metadata": {},
  "model": "gpt-5-2025-08-07",
  "object": "response",
  "output": [
    {
      "id": "rs_05de88